In [0]:
# Load all tables from restaurant_dev.bronze schema
bronze_tables = {
    "restaurants": spark.read.table("restaurant_dev.bronze.restaurants"),
    "customers": spark.read.table("restaurant_dev.bronze.customers"),
    "employees": spark.read.table("restaurant_dev.bronze.employees"),
    "menu_items": spark.read.table("restaurant_dev.bronze.menu_items"),
    "orders": spark.read.table("restaurant_dev.bronze.orders"),
    "order_details": spark.read.table("restaurant_dev.bronze.order_details"),
    "customer_reviews": spark.read.table("restaurant_dev.bronze.customer_reviews"),
    "daily_operations": spark.read.table("restaurant_dev.bronze.daily_operations"),
    "delivery_performance": spark.read.table("restaurant_dev.bronze.delivery_performance"),
    "inventory": spark.read.table("restaurant_dev.bronze.inventory"),
}

# Primary key definitions for each table
primary_keys = {
    "restaurants": ["restaurant_id"],
    "customers": ["customer_id"],
    "employees": ["employee_id"],
    "menu_items": ["item_id", "restaurant_id"],
    "orders": ["order_id"],
    "order_details": ["order_detail_id"],
    "customer_reviews": ["review_id"],
    "daily_operations": ["operation_id"],
    "delivery_performance": ["delivery_id"],
    "inventory": ["inventory_id"],
}

# Display table overview
print("=" * 80)
print("BRONZE LAYER TABLE OVERVIEW")
print("=" * 80)
for name, df in sorted(bronze_tables.items()):
    print(f"  {name:25s}  rows: {df.count():>10,}  cols: {len(df.columns):>3}")
print("=" * 80)

In [0]:
# ============================================================
# 1. NULL ANALYSIS
# ============================================================
from pyspark.sql.functions import col, count, when, isnan, isnull
from pyspark.sql.types import NumericType

print("=" * 80)
print("NULL ANALYSIS")
print("=" * 80)

null_summary_rows = []
for table_name, df in sorted(bronze_tables.items()):
    total_rows = df.count()
    for c in df.columns:
        is_numeric = isinstance(df.schema[c].dataType, NumericType)
        if is_numeric:
            null_count = df.filter(col(c).isNull() | isnan(col(c))).count()
        else:
            null_count = df.filter(col(c).isNull()).count()
        if null_count > 0:
            pct = (null_count / total_rows) * 100 if total_rows > 0 else 0
            null_summary_rows.append((table_name, c, null_count, round(pct, 2)))

if null_summary_rows:
    null_df = spark.createDataFrame(null_summary_rows, ["table", "column", "null_count", "null_pct"])
    print(f"\nFound nulls in {len(null_summary_rows)} columns:\n")
    display(null_df.orderBy(col("null_pct").desc()))
else:
    print("\nNo nulls found in any table.")

In [0]:
# ============================================================
# 2. DUPLICATES ANALYSIS
# ============================================================
from pyspark.sql.functions import col, count as spark_count

print("=" * 80)
print("DUPLICATES ANALYSIS")
print("=" * 80)

dup_summary_rows = []
for table_name, df in sorted(bronze_tables.items()):
    total_rows = df.count()
    distinct_rows = df.distinct().count()
    dup_rows = total_rows - distinct_rows

    # Check primary key duplicates
    pk = primary_keys.get(table_name, [])
    pk_dups = 0
    if pk:
        pk_total = df.select(*[col(c) for c in pk]).count()
        pk_distinct = df.select(*[col(c) for c in pk]).distinct().count()
        pk_dups = pk_total - pk_distinct

    dup_summary_rows.append((
        table_name,
        total_rows,
        dup_rows,
        round((dup_rows / total_rows) * 100, 2) if total_rows > 0 else 0,
        ", ".join(pk),
        pk_dups
    ))

dup_df = spark.createDataFrame(
    dup_summary_rows,
    ["table", "total_rows", "dup_rows", "dup_pct", "primary_key", "pk_dups"]
)
print("\nDuplicate rows and primary key violations per table:\n")
display(dup_df.orderBy(col("dup_rows").desc()))

In [0]:
# ============================================================
# 3. OUTLIER DETECTION (IQR Method)
# ============================================================
from pyspark.sql.functions import col, approx_count_distinct
from pyspark.sql.types import IntegerType, LongType, DoubleType, FloatType, DecimalType

print("=" * 80)
print("OUTLIER DETECTION (IQR Method)")
print("=" * 80)

numeric_types = (IntegerType, LongType, DoubleType, FloatType, DecimalType)
outlier_rows = []

for table_name, df in sorted(bronze_tables.items()):
    total_count = df.count()
    for c in df.columns:
        if isinstance(df.schema[c].dataType, numeric_types):
            # Skip ID-like columns with too many distinct values
            distinct_vals = df.select(c).distinct().count()
            if distinct_vals < 5 or distinct_vals > total_count * 0.9:
                continue

            q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.01)
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            outlier_count = df.filter((col(c) < lower_bound) | (col(c) > upper_bound)).count()
            if outlier_count > 0:
                outlier_pct = (outlier_count / total_count) * 100
                outlier_rows.append((
                    table_name, c, round(q1, 2), round(q3, 2),
                    round(lower_bound, 2), round(upper_bound, 2),
                    outlier_count, round(outlier_pct, 2)
                ))

if outlier_rows:
    outlier_df = spark.createDataFrame(
        outlier_rows,
        ["table", "column", "q1", "q3", "lower_bound", "upper_bound", "outlier_count", "outlier_pct"]
    )
    print(f"\nFound outliers in {len(outlier_rows)} numeric columns:\n")
    display(outlier_df.orderBy(col("outlier_count").desc()))
else:
    print("\nNo outliers detected.")

In [0]:
# ============================================================
# 4. STANDARDIZATION ISSUES
# ============================================================
from pyspark.sql.functions import col, trim, length, when, count as spark_count, lower, upper, regexp_replace
from pyspark.sql.types import StringType

print("=" * 80)
print("STANDARDIZATION ISSUES")
print("=" * 80)

issues = []

# Categorical columns to check for inconsistent values
categorical_checks = {
    "customers": ["customer_segment", "preferred_category", "preferred_order_type"],
    "orders": ["order_type", "payment_method", "order_status"],
    "employees": ["position", "employment_status"],
    "menu_items": ["category", "sub_category"],
    "delivery_performance": ["traffic_condition", "weather_condition", "delivery_status"],
    "customer_reviews": ["category"],
}

print("\n--- 4a. CATEGORICAL VALUE DISTRIBUTIONS ---")
for table_name, cat_cols in sorted(categorical_checks.items()):
    df = bronze_tables[table_name]
    for c in cat_cols:
        if c in df.columns:
            print(f"\n  [{table_name}.{c}]")
            df.groupBy(c).count().orderBy(col("count").desc()).show(truncate=False)

print("\n--- 4b. LEADING/TRAILING WHITESPACE IN STRING COLUMNS ---")
for table_name, df in sorted(bronze_tables.items()):
    for c in df.columns:
        if isinstance(df.schema[c].dataType, StringType):
            ws_count = df.filter(
                (col(c) != trim(col(c))) & col(c).isNotNull()
            ).count()
            if ws_count > 0:
                issues.append((table_name, c, "Whitespace", ws_count))

if issues:
    print(f"\nFound whitespace issues in {len(issues)} columns:")
    for t, c, issue, cnt in issues:
        print(f"  {t}.{c}: {cnt} rows with leading/trailing whitespace")
else:
    print("  No whitespace issues found.")

print("\n--- 4c. INCONSISTENT CASING IN CATEGORICAL COLUMNS ---")
for table_name, cat_cols in sorted(categorical_checks.items()):
    df = bronze_tables[table_name]
    for c in cat_cols:
        if c in df.columns:
            lower_distinct = df.select(lower(col(c)).alias("lc")).distinct().count()
            actual_distinct = df.select(col(c)).distinct().count()
            if actual_distinct > lower_distinct:
                print(f"  [{table_name}.{c}]: {actual_distinct} distinct values -> {lower_distinct} when lowercased (possible casing inconsistency)")
                df.groupBy(col(c)).count().orderBy(col("count").desc()).show(truncate=False)

print("\n--- 4d. DATA TYPE ANOMALIES ---")
# Check customer_id in orders/reviews (was inferred as double, should be int)
for table_name in ["orders", "customer_reviews"]:
    df = bronze_tables[table_name]
    if "customer_id" in df.columns:
        dtype = df.schema["customer_id"].dataType
        print(f"  [{table_name}.customer_id] dtype = {dtype}  (expected: integer)")

# Check for negative values where negatives shouldn't exist
negative_checks = {
    "orders": ["total_amount", "delivery_distance_miles", "delivery_time_mins"],
    "menu_items": ["price", "cost_to_make"],
    "daily_operations": ["daily_revenue", "total_customers_eat_in", "total_customers_delivery", "staff_on_shift"],
    "inventory": ["opening_stock", "closing_stock", "stock_value"],
    "customer_reviews": ["rating", "sentiment_score"],
    "employees": ["salary_per_hour", "performance_rating"],
}

print("\n--- 4e. NEGATIVE VALUES CHECK ---")
neg_issues = []
for table_name, num_cols in sorted(negative_checks.items()):
    df = bronze_tables[table_name]
    for c in num_cols:
        if c in df.columns:
            neg_count = df.filter(col(c) < 0).count()
            if neg_count > 0:
                neg_issues.append((table_name, c, neg_count))

if neg_issues:
    print(f"  Found negative values in {len(neg_issues)} columns:")
    for t, c, cnt in neg_issues:
        print(f"  {t}.{c}: {cnt} rows with negative values")
else:
    print("  No unexpected negative values found.")

print("\n" + "=" * 80)
print("STANDARDIZATION ANALYSIS COMPLETE")
print("=" * 80)

In [0]:
# ============================================================
# 5. CSV vs BRONZE DATA INTEGRITY COMPARISON
# ============================================================
from pyspark.sql.functions import col, count as spark_count, lit, min as fmin, max as fmax
from pyspark.sql.types import StructType, StructField

print("=" * 80)
print("CSV vs BRONZE DATA INTEGRITY COMPARISON")
print("=" * 80)

BASE_PATH = "/Volumes/restaurant_dev/london_dev_schema/raw/restaurant_data"

categories = [
    "restaurants", "customers", "employees", "menu_items", "orders",
    "order_details", "customer_reviews", "daily_operations",
    "delivery_performance", "inventory"
]

integrity_rows = []
value_mismatches = []

for cat in categories:
    csv_path = f"{BASE_PATH}/{cat}/"
    csv_df = spark.read.csv(csv_path, header=True, inferSchema=True)
    bronze_df = bronze_tables[cat]

    csv_count = csv_df.count()
    bronze_count = bronze_df.count()
    row_diff = bronze_count - csv_count

    # Compare column sets (bronze may have _rescued_data added by Auto Loader)
    csv_cols = set(csv_df.columns)
    bronze_cols = set(bronze_df.columns)
    extra_bronze_cols = sorted(bronze_cols - csv_cols)
    missing_bronze_cols = sorted(csv_cols - bronze_cols)

    # Compare data types for shared columns
    type_mismatches = []
    for c in sorted(csv_cols & bronze_cols):
        csv_type = csv_df.schema[c].dataType
        bronze_type = bronze_df.schema[c].dataType
        if str(csv_type) != str(bronze_type):
            type_mismatches.append((c, str(csv_type), str(bronze_type)))

    # Compare value-level integrity on a sample
    for c in sorted(csv_cols & bronze_cols):
        # Compare null counts
        csv_nulls = csv_df.filter(col(c).isNull()).count()
        bronze_nulls = bronze_df.filter(col(c).isNull()).count()
        if csv_nulls != bronze_nulls:
            value_mismatches.append((cat, c, "null_count", csv_nulls, bronze_nulls))

        # Compare min/max for numeric columns
        if isinstance(csv_df.schema[c].dataType, (IntegerType, LongType, DoubleType, FloatType, DecimalType)):
            csv_stats = csv_df.agg(fmin(col(c)), fmax(col(c))).collect()[0]
            bronze_stats = bronze_df.agg(fmin(col(c)), fmax(col(c))).collect()[0]
            csv_min, csv_max = csv_stats[0], csv_stats[1]
            bronze_min, bronze_max = bronze_stats[0], bronze_stats[1]
            if csv_min != bronze_min or csv_max != bronze_max:
                value_mismatches.append((cat, c, "min/max",
                    f"csv=[{csv_min},{csv_max}]", f"bronze=[{bronze_min},{bronze_max}]"))

    # Compare distinct counts for all shared columns (catches dedup or value changes)
    distinct_mismatches = []
    for c in sorted(csv_cols & bronze_cols):
        csv_distinct = csv_df.select(c).distinct().count()
        bronze_distinct = bronze_df.select(c).distinct().count()
        if csv_distinct != bronze_distinct:
            distinct_mismatches.append((c, csv_distinct, bronze_distinct))

    integrity_rows.append((
        cat,
        csv_count,
        bronze_count,
        row_diff,
        len(csv_cols),
        len(bronze_cols),
        ", ".join(extra_bronze_cols) if extra_bronze_cols else "none",
        ", ".join(missing_bronze_cols) if missing_bronze_cols else "none",
        len(type_mismatches),
        len(distinct_mismatches)
    ))

# Display summary table
print("\n--- 5a. ROW COUNT & SCHEMA COMPARISON ---")
integrity_df = spark.createDataFrame(
    integrity_rows,
    ["table", "csv_rows", "bronze_rows", "row_diff", "csv_cols", "bronze_cols",
     "extra_bronze_cols", "missing_in_bronze", "type_mismatches", "distinct_mismatches"]
)
display(integrity_df)

# Display type mismatches in detail
print("\n--- 5b. DATA TYPE MISMATCHES ---")
type_mismatch_rows = []
for cat in categories:
    csv_path = f"{BASE_PATH}/{cat}/"
    csv_df = spark.read.csv(csv_path, header=True, inferSchema=True)
    bronze_df = bronze_tables[cat]
    csv_cols = set(csv_df.columns)
    bronze_cols = set(bronze_df.columns)
    for c in sorted(csv_cols & bronze_cols):
        csv_type = str(csv_df.schema[c].dataType)
        bronze_type = str(bronze_df.schema[c].dataType)
        if csv_type != bronze_type:
            type_mismatch_rows.append((cat, c, csv_type, bronze_type))

if type_mismatch_rows:
    type_df = spark.createDataFrame(type_mismatch_rows, ["table", "column", "csv_type", "bronze_type"])
    display(type_df.orderBy(col("table")))
else:
    print("  No type mismatches found.")

# Display value-level mismatches
print("\n--- 5c. NULL COUNT & MIN/MAX MISMATCHES ---")
if value_mismatches:
    vm_df = spark.createDataFrame(
        value_mismatches,
        ["table", "column", "check", "csv_value", "bronze_value"]
    )
    print(f"\nFound {len(value_mismatches)} value-level mismatches:\n")
    display(vm_df)
else:
    print("  No null count or min/max mismatches found.")

# Display distinct count mismatches
print("\n--- 5d. DISTINCT COUNT MISMATCHES ---")
distinct_mismatch_rows = []
for cat in categories:
    csv_path = f"{BASE_PATH}/{cat}/"
    csv_df = spark.read.csv(csv_path, header=True, inferSchema=True)
    bronze_df = bronze_tables[cat]
    csv_cols = set(csv_df.columns)
    bronze_cols = set(bronze_df.columns)
    for c in sorted(csv_cols & bronze_cols):
        csv_distinct = csv_df.select(c).distinct().count()
        bronze_distinct = bronze_df.select(c).distinct().count()
        if csv_distinct != bronze_distinct:
            distinct_mismatch_rows.append((cat, c, csv_distinct, bronze_distinct))

if distinct_mismatch_rows:
    dm_df = spark.createDataFrame(
        distinct_mismatch_rows,
        ["table", "column", "csv_distinct", "bronze_distinct"]
    )
    print(f"\nFound {len(distinct_mismatch_rows)} distinct count mismatches:\n")
    display(dm_df)
else:
    print("  No distinct count mismatches found.")

print("\n" + "=" * 80)
print("DATA INTEGRITY COMPARISON COMPLETE")
print("=" * 80)

In [0]:
# ============================================================
# SILVER LAYER DATA QUALITY AUDIT
# ============================================================
# Load all tables from restaurant_dev.silver schema
silver_tables = {
    "restaurants": spark.read.table("restaurant_dev.silver.restaurants"),
    "customers": spark.read.table("restaurant_dev.silver.customers"),
    "employees": spark.read.table("restaurant_dev.silver.employees"),
    "menu_items": spark.read.table("restaurant_dev.silver.menu_items"),
    "orders": spark.read.table("restaurant_dev.silver.orders"),
    "order_details": spark.read.table("restaurant_dev.silver.order_details"),
    "customer_reviews": spark.read.table("restaurant_dev.silver.customer_reviews"),
    "daily_operations": spark.read.table("restaurant_dev.silver.daily_operations"),
    "delivery_performance": spark.read.table("restaurant_dev.silver.delivery_performance"),
    "inventory": spark.read.table("restaurant_dev.silver.inventory"),
}

silver_pks = {
    "restaurants": ["restaurant_id"],
    "customers": ["customer_id"],
    "employees": ["employee_id"],
    "menu_items": ["item_id", "restaurant_id"],
    "orders": ["order_id"],
    "order_details": ["order_detail_id"],
    "customer_reviews": ["review_id"],
    "daily_operations": ["restaurant_id", "date"],
    "delivery_performance": ["delivery_id"],
    "inventory": ["inventory_id"],
}

print("=" * 80)
print("SILVER LAYER TABLE OVERVIEW")
print("=" * 80)
for name, df in sorted(silver_tables.items()):
    print(f"  {name:25s}  rows: {df.count():>10,}  cols: {len(df.columns):>3}")
print("=" * 80)

In [0]:
# ============================================================
# SILVER: NULL ANALYSIS
# ============================================================
from pyspark.sql.functions import col, isnan
from pyspark.sql.types import NumericType

print("=" * 80)
print("SILVER LAYER - NULL ANALYSIS")
print("=" * 80)

silver_null_rows = []
for table_name, df in sorted(silver_tables.items()):
    total_rows = df.count()
    for c in df.columns:
        is_num = isinstance(df.schema[c].dataType, NumericType)
        if is_num:
            null_count = df.filter(col(c).isNull() | isnan(col(c))).count()
        else:
            null_count = df.filter(col(c).isNull()).count()
        if null_count > 0:
            pct = (null_count / total_rows) * 100 if total_rows > 0 else 0
            silver_null_rows.append((table_name, c, null_count, round(pct, 2)))

if silver_null_rows:
    silver_null_df = spark.createDataFrame(silver_null_rows, ["table", "column", "null_count", "null_pct"])
    print(f"\nFound nulls in {len(silver_null_rows)} columns:\n")
    display(silver_null_df.orderBy(col("null_pct").desc()))
else:
    print("\nNo nulls found in any silver table.")

In [0]:
# ============================================================
# SILVER: DUPLICATES ANALYSIS
# ============================================================
print("=" * 80)
print("SILVER LAYER - DUPLICATES ANALYSIS")
print("=" * 80)

silver_dup_rows = []
for table_name, df in sorted(silver_tables.items()):
    total = df.count()
    distinct = df.distinct().count()
    dup = total - distinct

    pk = silver_pks.get(table_name, [])
    pk_dups = 0
    if pk:
        pk_total = df.select(*[col(c) for c in pk]).count()
        pk_distinct = df.select(*[col(c) for c in pk]).distinct().count()
        pk_dups = pk_total - pk_distinct

    silver_dup_rows.append((
        table_name, total, dup,
        round((dup / total) * 100, 2) if total > 0 else 0,
        ", ".join(pk), pk_dups
    ))

silver_dup_df = spark.createDataFrame(
    silver_dup_rows,
    ["table", "total_rows", "dup_rows", "dup_pct", "primary_key", "pk_dups"]
)
print("\nDuplicate rows and primary key violations per table:\n")
display(silver_dup_df.orderBy(col("dup_rows").desc()))

In [0]:
# ============================================================
# SILVER: OUTLIER DETECTION (IQR Method)
# ============================================================
from pyspark.sql.types import IntegerType, LongType, DoubleType, FloatType, DecimalType

print("=" * 80)
print("SILVER LAYER - OUTLIER DETECTION (IQR Method)")
print("=" * 80)

numeric_types = (IntegerType, LongType, DoubleType, FloatType, DecimalType)
silver_outlier_rows = []

for table_name, df in sorted(silver_tables.items()):
    total_count = df.count()
    for c in df.columns:
        if isinstance(df.schema[c].dataType, numeric_types):
            distinct_vals = df.select(c).distinct().count()
            if distinct_vals < 5 or distinct_vals > total_count * 0.9:
                continue

            q1, q3 = df.approxQuantile(c, [0.25, 0.75], 0.01)
            iqr = q3 - q1
            lower_bound = q1 - 1.5 * iqr
            upper_bound = q3 + 1.5 * iqr

            outlier_count = df.filter((col(c) < lower_bound) | (col(c) > upper_bound)).count()
            if outlier_count > 0:
                outlier_pct = (outlier_count / total_count) * 100
                silver_outlier_rows.append((
                    table_name, c, round(q1, 2), round(q3, 2),
                    round(lower_bound, 2), round(upper_bound, 2),
                    outlier_count, round(outlier_pct, 2)
                ))

if silver_outlier_rows:
    silver_outlier_df = spark.createDataFrame(
        silver_outlier_rows,
        ["table", "column", "q1", "q3", "lower_bound", "upper_bound", "outlier_count", "outlier_pct"]
    )
    print(f"\nFound outliers in {len(silver_outlier_rows)} numeric columns:\n")
    display(silver_outlier_df.orderBy(col("outlier_count").desc()))
else:
    print("\nNo outliers detected in any silver table.")

In [0]:
# ============================================================
# SILVER: STANDARDIZATION CHECK
# ============================================================
from pyspark.sql.functions import trim, lower, StringType as StrType

print("=" * 80)
print("SILVER LAYER - STANDARDIZATION CHECK")
print("=" * 80)

# 1. Check for _rescued_data column (should be gone in silver)
print("\n--- 1. _rescued_data Column Check ---")
rescued_found = []
for table_name, df in sorted(silver_tables.items()):
    if "_rescued_data" in df.columns:
        rescued_found.append(table_name)
if rescued_found:
    print(f"  Found _rescued_data in: {rescued_found}")
else:
    print("  _rescued_data removed from all silver tables.")

# 2. Check customer_id type (should be int, not double)
print("\n--- 2. Data Type Verification ---")
for table_name in ["orders", "customer_reviews"]:
    df = silver_tables[table_name]
    if "customer_id" in df.columns:
        dtype = df.schema["customer_id"].dataType
        status = "OK" if "Integer" in str(dtype) else "STILL DOUBLE"
        print(f"  [{table_name}.customer_id] dtype = {dtype}  -> {status}")

# 3. Whitespace check on string columns
print("\n--- 3. Whitespace Check ---")
ws_issues = []
for table_name, df in sorted(silver_tables.items()):
    for c in df.columns:
        if isinstance(df.schema[c].dataType, StrType):
            ws_count = df.filter((col(c) != trim(col(c))) & col(c).isNotNull()).count()
            if ws_count > 0:
                ws_issues.append((table_name, c, ws_count))
if ws_issues:
    print(f"  Found whitespace issues in {len(ws_issues)} columns:")
    for t, c, cnt in ws_issues:
        print(f"    {t}.{c}: {cnt} rows")
else:
    print("  No whitespace issues found.")

# 4. Categorical value distributions (check for remaining inconsistencies)
print("\n--- 4. Categorical Value Distributions ---")
cat_checks = {
    "customers": ["customer_segment", "preferred_category", "preferred_order_type"],
    "orders": ["order_type", "payment_method", "order_status"],
    "employees": ["position", "employment_status"],
    "menu_items": ["category", "sub_category"],
    "delivery_performance": ["traffic_condition", "weather_condition", "delivery_status"],
    "customer_reviews": ["category"],
}
for table_name, cat_cols in sorted(cat_checks.items()):
    df = silver_tables[table_name]
    for c in cat_cols:
        if c in df.columns:
            print(f"\n  [{table_name}.{c}]")
            df.groupBy(c).count().orderBy(col("count").desc()).show(truncate=False)

# 5. Negative values check
print("\n--- 5. Negative Values Check ---")
neg_checks = {
    "orders": ["total_amount", "delivery_distance_miles", "delivery_time_mins"],
    "menu_items": ["price", "cost_to_make"],
    "daily_operations": ["daily_revenue", "total_customers_eat_in", "total_customers_delivery", "staff_on_shift"],
    "inventory": ["opening_stock", "closing_stock", "stock_value"],
    "customer_reviews": ["rating", "sentiment_score"],
    "employees": ["salary_per_hour", "performance_rating"],
}
neg_found = []
for table_name, num_cols in sorted(neg_checks.items()):
    df = silver_tables[table_name]
    for c in num_cols:
        if c in df.columns:
            neg_count = df.filter(col(c) < 0).count()
            if neg_count > 0:
                neg_found.append((table_name, c, neg_count))
if neg_found:
    print(f"  Found negative values in {len(neg_found)} columns:")
    for t, c, cnt in neg_found:
        print(f"    {t}.{c}: {cnt} rows")
else:
    print("  No unexpected negative values found.")

# 6. Compare row counts: bronze vs silver (check for dedup impact)
print("\n--- 6. Bronze vs Silver Row Count Comparison ---")
comparison_rows = []
for table_name in sorted(silver_tables.keys()):
    bronze_count = bronze_tables[table_name].count()
    silver_count = silver_tables[table_name].count()
    diff = bronze_count - silver_count
    diff_pct = round((diff / bronze_count) * 100, 2) if bronze_count > 0 else 0
    comparison_rows.append((table_name, bronze_count, silver_count, diff, diff_pct))

comparison_df = spark.createDataFrame(
    comparison_rows,
    ["table", "bronze_rows", "silver_rows", "rows_removed", "removed_pct"]
)
display(comparison_df.orderBy(col("rows_removed").desc()))

print("\n" + "=" * 80)
print("SILVER LAYER DATA QUALITY AUDIT COMPLETE")
print("=" * 80)